In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import math

In [2]:
# ----------------------------
# Paths
# ----------------------------

training_scripts_dir = Path.cwd()
project_root = training_scripts_dir.parent

data_dir = project_root / "Data"
recordings_dir = data_dir / "PNGs"

model_path = training_scripts_dir / "best.pt"

output_dir = project_root / "runs_tracking"
output_dir.mkdir(exist_ok=True)

print("Project root:", project_root)
print("Recordings dir:", recordings_dir)
print("Model:", model_path)
print("Output dir:", output_dir)

Project root: d:\2026\aspire
Recordings dir: d:\2026\aspire\Data\PNGs
Model: d:\2026\aspire\Training Scripts\best.pt
Output dir: d:\2026\aspire\runs_tracking


In [3]:
# ----------------------------
# Settings
# ----------------------------

# Set this to one recording folder name, or None to process all folders.
recording_name = None

# Example:
# recording_name = "Experiment_01"

image_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

fps = 6

conf_threshold = 0.25
iou_threshold = 0.5
imgsz = 1024

# Bubble size filter in pixels.
# These are equivalent diameters, preferably from YOLO masks.
min_diameter_px = 5
max_diameter_px = 300

# Event detection settings
event_distance_multiplier = 1.5
split_min_children = 2
merge_min_parents = 2

draw_masks = True
draw_boxes = True
draw_ids = True
draw_events = True

save_video = True
save_csv = True

In [4]:
# ----------------------------
# Helper functions
# ----------------------------

def get_recording_folders(recordings_dir, recording_name=None):
  if recording_name is not None:
    folder = recordings_dir / recording_name
    if not folder.exists():
      raise FileNotFoundError(f"Recording folder not found: {folder}")
    return [folder]

  folders = [p for p in recordings_dir.iterdir() if p.is_dir()]
  folders = sorted(folders)

  if len(folders) == 0:
    # If PNGs directly contains images, treat it as one recording.
    image_files = get_image_files(recordings_dir)
    if len(image_files) > 0:
      return [recordings_dir]

  return folders


def get_image_files(folder):
  files = []
  for ext in image_extensions:
    files.extend(folder.rglob(f"*{ext}"))
  return sorted(files)


def polygon_area(points):
  if points is None or len(points) < 3:
    return 0.0

  pts = np.asarray(points, dtype=np.float32)
  x = pts[:, 0]
  y = pts[:, 1]

  return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def equivalent_diameter_from_area(area_px):
  if area_px <= 0:
    return 0.0
  return 2.0 * math.sqrt(area_px / math.pi)


def box_inscribed_diameter(xyxy):
  x1, y1, x2, y2 = xyxy
  w = max(0.0, x2 - x1)
  h = max(0.0, y2 - y1)
  return min(w, h)


def centroid_from_box(xyxy):
  x1, y1, x2, y2 = xyxy
  return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)


def draw_transparent_mask(frame, polygon, alpha=0.35):
  if polygon is None or len(polygon) < 3:
    return frame

  overlay = frame.copy()
  pts = np.asarray(polygon, dtype=np.int32)
  cv2.fillPoly(overlay, [pts], (0, 255, 0))
  return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)


def draw_detection(frame, det):
  x1, y1, x2, y2 = [int(v) for v in det["xyxy"]]

  cx, cy = det["centroid"].astype(int)
  cx = int(cx)
  cy = int(cy)

  if draw_masks and det["mask_xy"] is not None:
    frame = draw_transparent_mask(frame, det["mask_xy"])

  if draw_boxes:
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

  if draw_ids:
    label = f'ID {det["track_id"]} | d={det["diameter_px"]:.1f}px'
    cv2.putText(
      frame,
      label,
      (x1, max(20, y1 - 8)),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.5,
      (0, 255, 0),
      2,
      cv2.LINE_AA
    )

  cv2.circle(frame, (cx, cy), 3, (0, 255, 255), -1)

  return frame

In [5]:
# ----------------------------
# Detection extraction
# ----------------------------

def extract_tracked_detections(result, frame_index, image_path):
  detections = []

  if result.boxes is None:
    return detections

  boxes = result.boxes

  if boxes.id is None:
    return detections

  xyxys = boxes.xyxy.cpu().numpy()
  track_ids = boxes.id.cpu().numpy().astype(int)
  confs = boxes.conf.cpu().numpy() if boxes.conf is not None else np.ones(len(xyxys))
  classes = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else np.zeros(len(xyxys), dtype=int)

  mask_polygons = [None] * len(xyxys)

  if result.masks is not None and result.masks.xy is not None:
    for i, poly in enumerate(result.masks.xy):
      if i < len(mask_polygons):
        mask_polygons[i] = poly

  for i, xyxy in enumerate(xyxys):
    mask_xy = mask_polygons[i]

    box_diameter = box_inscribed_diameter(xyxy)

    if mask_xy is not None and len(mask_xy) >= 3:
      area_px = polygon_area(mask_xy)
      diameter_px = equivalent_diameter_from_area(area_px)
      size_method = "mask_area"
    else:
      area_px = math.pi * (box_diameter / 2) ** 2
      diameter_px = box_diameter
      size_method = "box_inscribed_circle"

    if diameter_px < min_diameter_px or diameter_px > max_diameter_px:
      continue

    centroid = centroid_from_box(xyxy)

    detections.append({
      "frame": frame_index,
      "image_path": str(image_path),
      "track_id": int(track_ids[i]),
      "class_id": int(classes[i]),
      "confidence": float(confs[i]),
      "x1": float(xyxy[0]),
      "y1": float(xyxy[1]),
      "x2": float(xyxy[2]),
      "y2": float(xyxy[3]),
      "cx": float(centroid[0]),
      "cy": float(centroid[1]),
      "area_px": float(area_px),
      "diameter_px": float(diameter_px),
      "size_method": size_method,
      "xyxy": xyxy.astype(float),
      "centroid": centroid,
      "mask_xy": mask_xy
    })

  return detections

In [6]:
# ----------------------------
# Coalescence / splitting heuristic
# ----------------------------

def detect_events(prev_dets, curr_dets):
  events = []

  if len(prev_dets) == 0 or len(curr_dets) == 0:
    return events

  prev_to_curr = {}
  curr_to_prev = {}

  for p in prev_dets:
    p_id = p["track_id"]
    p_c = p["centroid"]
    p_d = max(p["diameter_px"], 1.0)

    for c in curr_dets:
      c_id = c["track_id"]
      c_c = c["centroid"]
      c_d = max(c["diameter_px"], 1.0)

      dist = np.linalg.norm(c_c - p_c)
      threshold = event_distance_multiplier * max(p_d, c_d)

      if dist <= threshold:
        prev_to_curr.setdefault(p_id, []).append(c_id)
        curr_to_prev.setdefault(c_id, []).append(p_id)

  # Splitting: one previous bubble maps near multiple current bubbles.
  for parent_id, child_ids in prev_to_curr.items():
    unique_child_ids = sorted(set(child_ids))

    if len(unique_child_ids) >= split_min_children:
      events.append({
        "event_type": "possible_split",
        "parent_track_ids": [parent_id],
        "child_track_ids": unique_child_ids
      })

  # Coalescence: multiple previous bubbles map near one current bubble.
  for child_id, parent_ids in curr_to_prev.items():
    unique_parent_ids = sorted(set(parent_ids))

    if len(unique_parent_ids) >= merge_min_parents:
      events.append({
        "event_type": "possible_coalescence",
        "parent_track_ids": unique_parent_ids,
        "child_track_ids": [child_id]
      })

  return events


def draw_event_labels(frame, events, curr_dets):
  id_to_det = {d["track_id"]: d for d in curr_dets}

  for event in events:
    event_type = event["event_type"]

    if event_type == "possible_split":
      text = "SPLIT?"
      color = (255, 0, 255)
      target_ids = event["child_track_ids"]

    elif event_type == "possible_coalescence":
      text = "MERGE?"
      color = (0, 0, 255)
      target_ids = event["child_track_ids"]

    else:
      continue

    for track_id in target_ids:
      if track_id not in id_to_det:
        continue

      det = id_to_det[track_id]
      cx, cy = det["centroid"].astype(int)

      cv2.putText(
        frame,
        text,
        (cx + 8, cy - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2,
        cv2.LINE_AA
      )

      cv2.circle(frame, (cx, cy), 12, color, 2)

  return frame

In [7]:
# ----------------------------
# Main tracking function
# ----------------------------

def process_recording(recording_folder, model):
  image_files = get_image_files(recording_folder)

  if len(image_files) == 0:
    print(f"No images found in {recording_folder}")
    return None

  recording_output_dir = output_dir / recording_folder.name
  recording_output_dir.mkdir(exist_ok=True)

  video_path = recording_output_dir / f"{recording_folder.name}_bytetrack_visualized.mp4"
  csv_path = recording_output_dir / f"{recording_folder.name}_tracks.csv"
  events_csv_path = recording_output_dir / f"{recording_folder.name}_events.csv"

  first_frame = cv2.imread(str(image_files[0]))
  if first_frame is None:
    print(f"Could not read first frame: {image_files[0]}")
    return None

  height, width = first_frame.shape[:2]

  writer = None
  if save_video:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(video_path), fourcc, fps, (width, height))

  all_rows = []
  all_events = []

  prev_dets = []

  # Important: persist=True keeps ByteTrack alive across frames.
  for frame_index, image_path in enumerate(tqdm(image_files, desc=f"Tracking {recording_folder.name}")):
    frame = cv2.imread(str(image_path))

    if frame is None:
      print(f"Could not read frame: {image_path}")
      continue

    results = model.track(
      source=frame,
      persist=True,
      tracker="bytetrack.yaml",
      conf=conf_threshold,
      iou=iou_threshold,
      imgsz=imgsz,
      verbose=False
    )

    result = results[0]
    curr_dets = extract_tracked_detections(result, frame_index, image_path)

    events = detect_events(prev_dets, curr_dets)

    for event in events:
      row = {
        "frame": frame_index,
        "event_type": event["event_type"],
        "parent_track_ids": ",".join(map(str, event["parent_track_ids"])),
        "child_track_ids": ",".join(map(str, event["child_track_ids"])),
        "image_path": str(image_path)
      }
      all_events.append(row)

    vis = frame.copy()

    for det in curr_dets:
      vis = draw_detection(vis, det)

      row = {
        key: value
        for key, value in det.items()
        if key not in ["xyxy", "centroid", "mask_xy"]
      }
      all_rows.append(row)

    if draw_events:
      vis = draw_event_labels(vis, events, curr_dets)

    frame_label = f"{recording_folder.name} | frame {frame_index + 1}/{len(image_files)}"
    cv2.putText(
      vis,
      frame_label,
      (20, 35),
      cv2.FONT_HERSHEY_SIMPLEX,
      1.0,
      (255, 255, 255),
      2,
      cv2.LINE_AA
    )

    if writer is not None:
      writer.write(vis)

    prev_dets = curr_dets

  if writer is not None:
    writer.release()

  tracks_df = pd.DataFrame(all_rows)
  events_df = pd.DataFrame(all_events)

  if save_csv:
    tracks_df.to_csv(csv_path, index=False)
    events_df.to_csv(events_csv_path, index=False)

  print(f"Done: {recording_folder.name}")
  print(f"Video: {video_path}")
  print(f"Tracks CSV: {csv_path}")
  print(f"Events CSV: {events_csv_path}")

  return {
    "recording": recording_folder.name,
    "video_path": video_path,
    "tracks_csv_path": csv_path,
    "events_csv_path": events_csv_path,
    "tracks_df": tracks_df,
    "events_df": events_df
  }

In [8]:
# ----------------------------
# Run ByteTrack visualizer
# ----------------------------

model = YOLO(str(model_path))

recording_folders = get_recording_folders(recordings_dir, recording_name)

print(f"Found {len(recording_folders)} recording folder(s):")
for folder in recording_folders:
  print(" -", folder.name)

outputs = []

for folder in recording_folders:
  output = process_recording(folder, model)
  if output is not None:
    outputs.append(output)

Found 5 recording folder(s):
 - 1.2mpm-6LPM-4fps-200us-dB20
 - 1.8mpm-3LPM-4fps-100us-30dB
 - 1.8mpm-6LPM-4fps-100us-30dB
 - 1.8mpm-6LPM-4fps-200us-20dB
 - 1.8mpm-6LPM-4fps-300us-20dB


Tracking 1.2mpm-6LPM-4fps-200us-dB20: 100%|██████████| 8/8 [00:21<00:00,  2.63s/it]


Done: 1.2mpm-6LPM-4fps-200us-dB20
Video: d:\2026\aspire\runs_tracking\1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-4fps-200us-dB20_bytetrack_visualized.mp4
Tracks CSV: d:\2026\aspire\runs_tracking\1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-4fps-200us-dB20_tracks.csv
Events CSV: d:\2026\aspire\runs_tracking\1.2mpm-6LPM-4fps-200us-dB20\1.2mpm-6LPM-4fps-200us-dB20_events.csv


Tracking 1.8mpm-3LPM-4fps-100us-30dB: 100%|██████████| 10/10 [00:09<00:00,  1.08it/s]


Done: 1.8mpm-3LPM-4fps-100us-30dB
Video: d:\2026\aspire\runs_tracking\1.8mpm-3LPM-4fps-100us-30dB\1.8mpm-3LPM-4fps-100us-30dB_bytetrack_visualized.mp4
Tracks CSV: d:\2026\aspire\runs_tracking\1.8mpm-3LPM-4fps-100us-30dB\1.8mpm-3LPM-4fps-100us-30dB_tracks.csv
Events CSV: d:\2026\aspire\runs_tracking\1.8mpm-3LPM-4fps-100us-30dB\1.8mpm-3LPM-4fps-100us-30dB_events.csv


...

Done: 1.8mpm-6LPM-4fps-100us-30dB
Video: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-100us-30dB\1.8mpm-6LPM-4fps-100us-30dB_bytetrack_visualized.mp4
Tracks CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-100us-30dB\1.8mpm-6LPM-4fps-100us-30dB_tracks.csv
Events CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-100us-30dB\1.8mpm-6LPM-4fps-100us-30dB_events.csv


Tracking 1.8mpm-6LPM-4fps-200us-20dB: 100%|██████████| 12/12 [00:10<00:00,  1.11it/s]


Done: 1.8mpm-6LPM-4fps-200us-20dB
Video: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-200us-20dB\1.8mpm-6LPM-4fps-200us-20dB_bytetrack_visualized.mp4
Tracks CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-200us-20dB\1.8mpm-6LPM-4fps-200us-20dB_tracks.csv
Events CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-200us-20dB\1.8mpm-6LPM-4fps-200us-20dB_events.csv


Tracking 1.8mpm-6LPM-4fps-300us-20dB: 100%|██████████| 12/12 [00:11<00:00,  1.04it/s]

Done: 1.8mpm-6LPM-4fps-300us-20dB
Video: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-300us-20dB\1.8mpm-6LPM-4fps-300us-20dB_bytetrack_visualized.mp4
Tracks CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-300us-20dB\1.8mpm-6LPM-4fps-300us-20dB_tracks.csv
Events CSV: d:\2026\aspire\runs_tracking\1.8mpm-6LPM-4fps-300us-20dB\1.8mpm-6LPM-4fps-300us-20dB_events.csv


In [9]:
# ----------------------------
# Preview tracking table
# ----------------------------

if len(outputs) > 0:
  display(outputs[0]["tracks_df"].head())
  display(outputs[0]["events_df"].head())

,frame,image_path,track_id,class_id,confidence,x1,y1,x2,y2,cx,cy,area_px,diameter_px,size_method
0,0,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...,1,0,0.922152,2712.562500,1805.685303,2746.484375,1833.546753,2729.523438,1819.615967,756.0,31.025298,mask_area
1,0,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...,2,0,0.912595,4205.259766,3662.186035,4248.842773,3698.835449,4227.051270,3680.510742,1428.0,42.640193,mask_area
2,0,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...,3,0,0.866025,4664.112305,3181.569824,4697.101562,3216.567139,4680.606934,3199.068359,528.0,25.928179,mask_area
3,0,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...,4,0,0.854625,1777.553955,780.872192,1816.911987,810.561951,1797.232910,795.717041,1511.0,43.861884,mask_area
4,0,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...,5,0,0.852516,4410.065430,1067.906128,4452.164551,1092.572144,4431.115234,1080.239136,1134.0,37.998074,mask_area


,frame,event_type,parent_track_ids,child_track_ids,image_path
0,1,possible_coalescence,"173,205",205,d:\2026\aspire\Data\PNGs\1.2mpm-6LPM-4fps-200u...
